# Question 1 – Genetic Algorithm for the Traveling Salesperson Problem (TSP)

**From AI Lab Project (CSC462 - Artificial Intelligence):**

> The Traveling Salesperson Problem (TSP) is a classic NP-hard optimization problem in 
> which a salesperson must visit a set of cities exactly once, return to the starting point, and 
> minimize the total travel distance. For large numbers of cities, brute-force or exhaustive 
> search approaches become computationally infeasible. Genetic Algorithms (GAs) provide 
> an effective heuristic method to obtain near-optimal solutions within reasonable time. To 
> implement a GA-based solution for TSP, the following components are required:
>
> 1. Population Initialization: Generate an initial random population of tours.
> 2. Chromosome Representation: Represent a tour (sequence of cities) as a 
>    permutation of city names.
> 3. Fitness Function: Calculate the fitness of each tour (chromosome) based on its 
>    total distance.
> 4. Selection: Implement a Tournament selection mechanism to choose parents for 
>    reproduction. Preserve a certain number of the best individuals from one 
>    generation to the next.
> 5. Crossover: Implement a TSP-specific Partially Mapped Crossover (PMX) 
>    crossover operator to create offspring.
> 6. Mutation: Implement a Swap mutation operator to introduce diversity into the 
>    population.
> 7. Termination Criteria: Stop the GA after a fixed number of generations or if the 
>    fitness improvement stops.
> 8. Output: Display the best-found tour and its total distance.


## Step 1 — Import Libraries & Define TSP Cities
This cell imports the required libraries and defines the TSP cities.

In [ ]:
print("STEP 1: Loading libraries and defining city coordinates...")

import random
import math

# Define coordinates of each city
cities = {
    "A": (10, 20),
    "B": (20, 80),
    "C": (80, 90),
    "D": (90, 40),
    "E": (50, 10),
    "F": (30, 50),
    "G": (60, 70),
    "H": (70, 20),
    "I": (15, 60),
    "J": (40, 90),
}

print("Cities and their coordinates:")
for name, coord in cities.items():
    print(f"  {name}: {coord}")

STEP 1: Loading libraries and defining city coordinates...
Cities and their coordinates:
  A: (10, 20)
  B: (20, 80)
  C: (80, 90)
  D: (90, 40)
  E: (50, 10)
  F: (30, 50)
  G: (60, 70)
  H: (70, 20)
  I: (15, 60)
  J: (40, 90)


## Step 2 — Distance Matrix, Tour Length & Fitness
This cell defines helper functions used throughout the GA. The **actual data** printed later will come from inside the GA.

In [ ]:
print("STEP 2: Defining distance matrix, tour length, and fitness function...")

def euclidean_distance(p1, p2):
    # Euclidean distance between two (x,y) points.
    return math.dist(p1, p2)

def compute_distance_matrix(cities):
    # Build a distance matrix between all cities.
    names = list(cities.keys())
    n = len(names)
    dist = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                dist[i][j] = euclidean_distance(cities[names[i]], cities[names[j]])
    return names, dist

def tour_length(tour, dist, name_to_index):
    # Total distance of a tour (including return to start).
    total = 0.0
    for i in range(len(tour)):
        a = tour[i]
        b = tour[(i + 1) % len(tour)]
        total += dist[name_to_index[a]][name_to_index[b]]
    return total

def fitness_from_length(L):
    # Fitness = 1 / (1 + distance) so shorter tours have higher fitness.
    return 1.0 / (1.0 + L)

print("Helper functions defined (euclidean_distance, compute_distance_matrix, tour_length, fitness_from_length).")

STEP 2: Defining distance matrix, tour length, and fitness function...
Helper functions defined (euclidean_distance, compute_distance_matrix, tour_length, fitness_from_length).


## Step 3 — Population Initialization & Tournament Selection
This cell creates random tours and defines how parents are selected. The **real values** used will be printed from inside the GA.

In [ ]:
print("STEP 3: Defining population initialization and tournament selection...")

def initial_population(city_names, pop_size):
    # Create an initial random population of tours (permutations).
    base = city_names[:]
    population = []
    for _ in range(pop_size):
        chrom = base[:]
        random.shuffle(chrom)
        population.append(chrom)
    return population

def tournament_select(population, lengths, k):
    # Tournament selection: return best of k randomly chosen individuals.
    indices = random.sample(range(len(population)), k)
    best_index = min(indices, key=lambda idx: lengths[idx])
    return population[best_index][:]

print("Functions initial_population and tournament_select are defined.")

STEP 3: Defining population initialization and tournament selection...
Functions initial_population and tournament_select are defined.


## Step 4 — PMX Crossover & Swap Mutation
This cell implements GA variation operators. Their **real effect** will be shown via prints from inside the GA loop.

In [ ]:
print("STEP 4: Defining PMX crossover and swap mutation...")

def pmx_child(parent1, parent2, c1, c2):
    # Build one child using Partially Mapped Crossover (PMX).
    size = len(parent1)
    child = [None] * size

    # 1) Copy middle segment from parent1
    child[c1:c2+1] = parent1[c1:c2+1]

    # 2) Mapping from parent2
    for i in range(c1, c2+1):
        if parent2[i] not in child:
            pos = i
            while True:
                value = parent1[pos]
                pos = parent2.index(value)
                if child[pos] is None:
                    child[pos] = parent2[i]
                    break

    # 3) Fill remaining positions from parent2
    for i in range(size):
        if child[i] is None:
            child[i] = parent2[i]

    return child

def pmx_crossover(parent1, parent2):
    # Partially Mapped Crossover (PMX) for permutations. Returns two children.
    size = len(parent1)
    c1, c2 = sorted(random.sample(range(size), 2))
    child1 = pmx_child(parent1, parent2, c1, c2)
    child2 = pmx_child(parent2, parent1, c1, c2)
    return child1, child2

def swap_mutation(tour, mutation_rate):
    # Swap mutation: with given probability, swap two random cities.
    tour = tour[:]
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(tour)), 2)
        tour[i], tour[j] = tour[j], tour[i]
    return tour

print("Functions pmx_crossover and swap_mutation are defined.")

STEP 4: Defining PMX crossover and swap mutation...
Functions pmx_crossover and swap_mutation are defined.


## Step 5 — Main Genetic Algorithm
This cell defines the full GA. All printed distances, fitness values, parents, and chromosomes come from the **actual data used** by the algorithm.

In [ ]:
print("STEP 5: Defining the main genetic_algorithm_tsp function with detailed logging...")

def genetic_algorithm_tsp(
    cities,
    pop_size=30,
    generations=50,
    tournament_k=3,
    crossover_rate=0.9,
    mutation_rate=0.1,
    elitism_size=2,
    patience=20,
):
    """GA for TSP using tournament selection, PMX, swap mutation, and elitism.

    All printed values are from the REAL population used by the algorithm."""

    # Compute distance matrix once
    city_names, dist = compute_distance_matrix(cities)
    name_to_index = {name: i for i, name in enumerate(city_names)}

    print("\n--- INITIALIZATION ---")
    print("City order used internally:", city_names)
    print("Showing first 3 rows of the distance matrix:")
    for row in dist[:3]:
        print("  ", row)

    # 1) Initial population
    population = initial_population(city_names, pop_size)
    lengths = [tour_length(ch, dist, name_to_index) for ch in population]

    print("\nInitial population (showing all tours with distance and fitness):")
    for i, chrom in enumerate(population):
        L = lengths[i]
        F = fitness_from_length(L)
        print(f"  Chromosome {i}: {chrom} | Distance = {L:.2f}, Fitness = {F:.6f}")

    # 2) Track best solution
    best_idx = min(range(pop_size), key=lambda idx: lengths[idx])
    best_tour = population[best_idx][:]
    best_length = lengths[best_idx]
    best_fitness = fitness_from_length(best_length)
    no_improve = 0

    print("\nBest individual in initial population:")
    print(f"  Tour: {best_tour}")
    print(f"  Distance: {best_length:.2f}, Fitness: {best_fitness:.6f}")

    # GA main loop
    for gen in range(generations):
        print(f"\n=== GENERATION {gen} ===")

        new_population = []

        # --- Elitism: keep top 'elitism_size' individuals ---
        elite_indices = sorted(range(len(population)), key=lambda idx: lengths[idx])[:elitism_size]
        print("Elite individuals carried over:")
        for e_i in elite_indices:
            print(f"  Elite {e_i}: {population[e_i]} | Distance = {lengths[e_i]:.2f}")

        for idx in elite_indices:
            new_population.append(population[idx][:])

        # --- Create the rest of the new population ---
        mating_logged = False  # we will log details for the first mating event only (per generation)
        while len(new_population) < pop_size:
            # Selection
            parent1 = tournament_select(population, lengths, tournament_k)
            parent2 = tournament_select(population, lengths, tournament_k)

            # Crossover
            if random.random() < crossover_rate:
                child1, child2 = pmx_crossover(parent1, parent2)
            else:
                child1, child2 = parent1[:], parent2[:]

            # Mutation
            child1 = swap_mutation(child1, mutation_rate)
            child2 = swap_mutation(child2, mutation_rate)

            # Log details for the first pair per generation
            if not mating_logged:
                print("\nExample mating in this generation:")
                print("  Parent 1:", parent1)
                print("  Parent 2:", parent2)
                print("  Child 1 (after PMX + mutation):", child1)
                print("  Child 2 (after PMX + mutation):", child2)
                mating_logged = True

            new_population.append(child1)
            if len(new_population) < pop_size:
                new_population.append(child2)

        # Move to next generation
        population = new_population
        lengths = [tour_length(ch, dist, name_to_index) for ch in population]

        # Compute stats for this generation
        gen_best_idx = min(range(pop_size), key=lambda idx: lengths[idx])
        gen_best_length = lengths[gen_best_idx]
        gen_best_fitness = fitness_from_length(gen_best_length)
        avg_length = sum(lengths) / len(lengths)

        print("\nSummary of this generation:")
        print(f"  Best distance: {gen_best_length:.2f}")
        print(f"  Best fitness:  {gen_best_fitness:.6f}")
        print(f"  Average distance: {avg_length:.2f}")

        # Update global best
        if gen_best_length < best_length - 1e-9:
            best_length = gen_best_length
            best_tour = population[gen_best_idx][:]
            best_fitness = gen_best_fitness
            no_improve = 0
            print("  --> New global best found!")
        else:
            no_improve += 1
            print(f"  No improvement in this generation. (No-improve count = {no_improve})")

        # Termination: no improvement for 'patience' generations
        if no_improve >= patience:
            print(f"\nStopping early: no improvement for {patience} generations.")
            break

    print("\n--- GA FINISHED ---")
    print("Best tour found during run:")
    print("  ", " -> ".join(best_tour) + " -> " + best_tour[0])
    print(f"Best distance: {best_length:.2f}")
    print(f"Best fitness:  {best_fitness:.6f}")

    return best_tour, best_length, best_fitness

print("genetic_algorithm_tsp is now defined with detailed logging.")

STEP 5: Defining the main genetic_algorithm_tsp function with detailed logging...
genetic_algorithm_tsp is now defined with detailed REAL logging.


## Step 6 — Run the Genetic Algorithm
This cell runs the GA and the **printed values are exactly the ones used** during optimization.

In [6]:
print("STEP 6: Running the genetic algorithm on the TSP instance...")

# For reproducibility
random.seed(42)

best_tour, best_distance, best_fit = genetic_algorithm_tsp(
    cities,
    pop_size=20,      # smaller pop & generations to keep output readable
    generations=30,
    tournament_k=3,
    crossover_rate=0.9,
    mutation_rate=0.1,
    elitism_size=2,
    patience=10,
)

print("\nFINAL RESULT (returned by function):")
print("  Best tour:", " -> ".join(best_tour) + " -> " + best_tour[0])
print(f"  Total distance: {best_distance:.2f}")
print(f"  Fitness: {best_fit:.6f}")

STEP 6: Running the genetic algorithm on the TSP instance...

--- INITIALIZATION ---
City order used internally: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
Showing first 3 rows of the distance matrix:
   [0.0, 60.8276253029822, 98.99494936611666, 82.46211251235322, 41.23105625617661, 36.05551275463989, 70.71067811865476, 60.0, 40.311288741492746, 76.15773105863909]
   [60.8276253029822, 0.0, 60.8276253029822, 80.62257748298549, 76.15773105863909, 31.622776601683793, 41.23105625617661, 78.10249675906654, 20.615528128088304, 22.360679774997898]
   [98.99494936611666, 60.8276253029822, 0.0, 50.99019513592785, 85.44003745317531, 64.03124237432849, 28.284271247461902, 70.71067811865476, 71.58910531638176, 40.0]

Initial population (showing all tours with distance and fitness):
  Chromosome 0: ['H', 'D', 'C', 'I', 'F', 'G', 'J', 'E', 'A', 'B'] | Distance = 494.01, Fitness = 0.002020
  Chromosome 1: ['D', 'F', 'C', 'E', 'B', 'I', 'H', 'A', 'G', 'J'] | Distance = 604.79, Fitness = 0.00